**1. Data acquisition and intuitive analysis**

*For this task I started by downloading the data I needed.*
I did it as describe but I just downloaded two folders (after setting the env variable CSV_URL):
- aws s3 sync --no-sign-request $CSV_URL/feature_wo_messages/ .
- aws s3 sync --no-sign-request $CSV_URL/messages .

I then after checking the data, I observed what I considered as abnomally:
ie, in file message_143 there are users reposting stats on most mentionned tickers
443015032,@RyGuyBoom lol
**443015033,"Top 10 #WallstreetBets Stocks
Updated Hourly
$AMZN : 158 mentions
$TSLA : 70 mentions
$XOM : 52 mentions
$BABA : 40 mentions
$R : 30 mentions
$AMD : 29 mentions
$GME : 26 mentions
$BBBY : 22 mentions
$AAPL : 20 mentions
$OXY : 17 mentions
TradeUI.com/wallstreetbets"**
443015034,$CEI why tf is crude oil dropping?

in the file feature_wo_message173 this message was tagged as bullish while it just contain some stats without sentiment in it.

this kind of observations on bullish or bearish sentiment lead me to try another classification of the message.

Will using another classification method improve the results?

**2. Market sentiment classification**

There are two methods I want to try in this phase:
- *A. Classify using finbert*
- *B. depending on the result, I want to use an LLM for labelisation and then fine tune finBERT* 

to evaluate their performance I will use data related to bitcoin, as the periodic bullrun and bearmarket make it easy to manually classify.

For evaluation I'll check how the message sent on the social media during the bullruns (2017-09-30/2017-12-16; 2020-10-03/2021-03-13) are anoted by the finbert and I'll try the same analysis on a local llm foundation model. I will do the same on bear market periods (~2017-12-17/2019-02-01; 2021-11-20/2022-12-31)
The intuition behind this is that the amount of bullish VS bearish messages should be very high or low depending on these periods.



In [1]:
#import of finBert transformers
from transformers import pipeline
classifier = pipeline("sentiment-analysis", model="ProsusAI/finbert")


c:\Users\xis\Documents\dossier ecole\cours\nlp\projet\stock_market_prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 261.29it/s]


*first attempts of classification on a known bearish sentence*

In [2]:
result = classifier("@Berkshireannhathaway 175?!!!  Whatever happened to 100,000 per share?  (Your post perfectly illustrates why it can never moass.  Too many bag holders)")
print(result)

[{'label': 'neutral', 'score': 0.8756725788116455}]


In [ ]:
result = classifier('''Top 10 #WallstreetBets Stocks
Updated Hourly
$AMZN : 158 mentions
$TSLA : 70 mentions
$XOM : 52 mentions
$BABA : 40 mentions
$R : 30 mentions
$AMD : 29 mentions
$GME : 26 mentions
$BBBY : 22 mentions
$AAPL : 20 mentions
$OXY : 17 mentions
TradeUI.com/wallstreetbets''')
print(result)

In [ ]:
result = classifier('''$EVRI New Press Release:  
  
EVERI HOLDINGS JOINS THE U.S. ARMY'S PARTNERSHIP FOR YOUTH SUCCESS PROGRAM 
  
Everi Holdings Inc. (NYSE: EVRI) (&quot;Everi&quot; or &quot;the Company&quot;), a premier provider of land-based and digital casino gaming content and products, financial technology, and player loyalty solutions, announced today an agreement between the Company and the U.S. Army whereby Everi will enter the Army's Partnership for Youth Success (&quot;PaYS&quot;) Program. The PaYS Program is a strategic partnership between the U.S. Army and a cross section of corporations, companies, and public sector agencies. PaYS partners, such as Everi, guarantee soldiers an interview and possible employment after the Army. This unique program is part of the Army's effort to partner with America's business community and provides Amer (...) 
  
https://webchronicletoday.com?p=228282''')
print(result)

*The finBert seems to work even tough for an bearish message the result seemed neutral*
now are going to select in the data the files corresponding to a BTC bullrun period (2017-09-30/2017-12-16) and create a dataframe only containing msg with BTC tickers
the msg ids corresponding to those date are: 100000000-105355476

*Checking the data shows that there is a big gap between 2012 and begining of november 2017 in the available data. thankfully, we have a enough data to do our comparatives cheks on finBert and llm classification results.* 

In [4]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/msg/msg_000.csv', sep=',')
display(df.head())
display(df.size)


In [ ]:
display(df.head(1000))

In [13]:
#df['message_id'].astype('int')
#df = df.dropna(subset=['message_id'])
#df = df[df['message_id'].str.fullmatch(r'\d+')].reset_index(drop=True)
#après de nombreux test:
files = ['../data/msg/msg_000.csv', '../data/msg/msg_001.csv']
df = pd.concat([pd.read_csv(f, dtype={'message_id': str}, on_bad_lines='error') for f in files], ignore_index=True)

#je n'arrive pas a me servir de la variable message ID comme une valeur entiere car les données contiennent des retours chariot problématique
#je vais donc ouvrir le fichier de manière raw et remplacer le retour chariot par un espace
'''import re
def clean_idx(fichier):
    with open(fichier, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    fixed = []
    for line in lines:
        line = line.rstrip('\n')
        if line and line[0].isdigit():
            fixed.append(line)
        elif fixed:
            fixed[-1] += '\n' + line

    with open(fichier + 'fixed', 'w', encoding='utf-8') as f:
        f.write('\n'.join(fixed))
'''

C:\Users\xis\AppData\Local\Temp\ipykernel_2600\400151334.py:6: DtypeWarning: Columns (0: message_body) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f, dtype={'message_id': str}, on_bad_lines='error') for f in files], ignore_index=True)


"import re\ndef clean_idx(fichier):\n    with open(fichier, 'r', encoding='utf-8', errors='replace') as f:\n        lines = f.readlines()\n\n    fixed = []\n    for line in lines:\n        line = line.rstrip('\n')\n        if line and line[0].isdigit():\n            fixed.append(line)\n        elif fixed:\n            fixed[-1] += '\n' + line\n\n    with open(fichier + 'fixed', 'w', encoding='utf-8') as f:\n        f.write('\n'.join(fixed))\n"

In [14]:
# Trouver les valeurs qui ne sont PAS de purs entiers
mask = ~df['message_id'].str.match(r'^\d+$')
print(df[mask][['message_id']].to_string())
print(f"\n{mask.sum()} lignes avec un message_id non-entier")

                                                                                    message_id
137152                                                                                  fixing
158550                                                          consumer finance in Brazil $GS
158551                                                                    http://t.co/h2BK9Xrf
179155                                        From The Nasdaq 100 ($NDX)  http://t.co/lM028wGo
334616                                                                    http://t.co/RzOLQhVd
345347                                               strength of resistance at those levels...
355785                                                                    http://t.co/wt5bX8Ts
524249                                                                    http://t.co/RrklUrTx
524367                                                                    http://t.co/3AjKPzlN
786432                                            

In [ ]:
df = df[df['message_id'].str.fullmatch(r'\d+', na=False)].reset_index(drop=True)
df = df.dropna(subset=['message_id'])


In [ ]:
df['message_id'] = df['message_id'].astype('int')


In [ ]:
begin_Bull_run = 100000000
end_bull_run = 105355476


In [ ]:
df = df[(df['message_id'] >= begin_Bull_run) & (df['message_id'] <= end_bull_run)].reset_index(drop=True)
df.head()


In [ ]:
df = df[df['message_body'].str.contains(r'\$btc', case=False)].reset_index(drop=True)
df.head()

In [ ]:
df['message_body']=df['message_body'].str.replace("&#39;","'")
df.head()

**Now comes the classification into bullish, bearish or neutral**

In [ ]:
results = classifier(df['message_body'].tolist())
df['sentiment'] = [r['label'] for r in results]
df['score'] = [r['score'] for r in results]


In [ ]:
#save the work
df.to_csv("../data/finBer_classification_Bullish.csv", index=False)

## Pour faciliter la suite, utiliser les fichiers déjà préprocessé

In [3]:
df_processed_btx_BR = pd.read_csv("../data/finBer_classification_Bullish.csv").reset_index(drop=True)
df_processed_btx_BR.head(20)
df_processed_btx_BR.shape()

NameError: name 'pd' is not defined

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#df['sentiment_num'] = df['sentiment'].map({'positive': 1, 'neutral': 0, 'negative': -1})
#plt.scatter(df['message_id'],df['sentiment_num']*df['score']+df['sentiment_num'], s=df['score']*3)
df_processed_btx_BR['sentiment'].value_counts().plot(kind='bar')

These results are not the one expected, the model seems to perform poorly on detecting the bullish message.

To confirm this I am going to check the classification made by the users themselves and try a classification with a locally installed frontier Model.



In [ ]:
#lets start with the users classification
#we will load the files containing the info regarding the bull run
df_trend_from_users = pd.concat([
    pd.read_csv('../data/msg_feature/feature_wo_messages_000.csv'),
    pd.read_csv('../data/msg_feature/feature_wo_messages_001.csv'),
    pd.read_csv('../data/msg_feature/feature_wo_messages_002.csv'),
    ], ignore_index=True)


In [ ]:
#after that we will run head() to check what's inside
df_trend_from_users.head()

In [ ]:
#we will now clean the msg out of the scope of 2017 bullrun and not related to BTX
df_trend_from_users = df_trend_from_users[(df_trend_from_users['message_id']>=begin_Bull_run) & (df_trend_from_users['message_id']<=end_bull_run)].reset_index(drop=True)
df_trend_from_users.head()

In [ ]:
#filter msg_feature on $btc ticker
df_trend_from_users = df_trend_from_users[df_trend_from_users['symbol_list'].apply(lambda x: "BTC.X" in x)].reset_index(drop=True)
df_trend_from_users.head(3000)


In [ ]:
df_trend_from_users['sentiment'] = df_trend_from_users['sentiment'].fillna('not_labeled')
df_trend_from_users['sentiment'].value_counts().plot(kind='bar')


In [ ]:
print(df_processed_btx_BR.size) # 259156
print(df_trend_from_users.size) # 686224
df_trend_from_users.to_csv("../data/BTC_Sentiment_from_feature.csv", index=False)
# there is a big gap in the data between the two df because I droped all the message containing
# more lines as there were not fiting in my df correctly.

#### later on we can try to use the data in the df_trend for a better preprocessing
for now i'll just continue the base of my system

**This part on one message works good, but I moved the global processing in the same file as for the finetuning in the notebook bycolab**

In [ ]:
#do it once only
#!ollama pull qwen2.5:7b


In [ ]:
!ollama list
#!ollama run qwen2.5:7b "Classify: NVDA earnings crushed estimates. Answer BULLISH, BEARISH or NEUTRAL:"
#!ollama run phi3:latest "you are a financial analyst,Classify: *NVDA earnings crushed estimates.* Answer BULLISH, BEARISH or NEUTRAL:"
#!taskkill /F /IM ollama.exe 
#because it took too long


In [3]:
from openai import OpenAI
import requests
requests.get("http://127.0.0.1:11434").content


b'Ollama is running'

In [ ]:
#download the model I selected
#!ollama pull phi4-mini already done


In [5]:
#add the variables and tests
ollama_url = 'http://127.0.0.1:11434/v1'
ollama = OpenAI(base_url=ollama_url, api_key='ollama')

In [ ]:
err_count_1 = 0
err_count_11 = 0
err_count_12 = 0
for i in range(300):
    response_phi = ollama.chat.completions.create(model='phi4-mini', messages=[{"role": "user",  "content":"classify this message as a bullish, bearish or neutral, when the classification is done sumarize your answer in ONLY one word 'bearish', 'bullish' or 'neutral': *@Berkshireannhathaway 175?!!!  Whatever happened to 100,000 per share?  (Your post perfectly illustrates why it can never moass.  Too many bag holders*"}])
    if (response_phi.choices[0].message.content != 'bearish'):
        err_count_1 += 1
        if (response_phi.choices[0].message.content.lower() != 'bearish'):
            err_count_11 += 1
        if (len(response_phi.choices[0].message.content) != len("bearish")):
            err_count_12 += 1

In [ ]:
err_count_2 = 0
err_count_21 = 0
err_count_22 = 0

for i in range(300):
    response_qwen = ollama.chat.completions.create(model='qwen2.5:7b', messages=[{"role": "user", "content":"classify this message as a bullish, bearish or neutral, when the classification is done sumarize your answer in just one word 'bearish', 'bullish' or 'neutral': *@Berkshireannhathaway 175?!!!  Whatever happened to 100,000 per share?  (Your post perfectly illustrates why it can never moass.  Too many bag holders*"}])
    if (response_qwen.choices[0].message.content != 'bearish'):
        err_count_2 += 1
        if (response_qwen.choices[0].message.content.lower() != 'bearish'):
            err_count_21 += 1
        if (len(response_qwen.choices[0].message.content) != len("bearish")):
            err_count_22 += 1

In [ ]:
err_count_3 = 0
err_count_31 = 0
err_count_32 = 0
for i in range(300):
    response_llam = ollama.chat.completions.create(model='llama3.1:8b', messages=[{"role": "user", "content":"classify this message as a bullish, bearish or neutral, when the classification is done sumarize your answer in just one word 'bearish', 'bullish' or 'neutral': *@Berkshireannhathaway 175?!!!  Whatever happened to 100,000 per share?  (Your post perfectly illustrates why it can never moass.  Too many bag holders*"}])
    if (response_llam.choices[0].message.content != 'bearish'):
        err_count_3 += 1
        if (response_llam.choices[0].message.content.lower() != 'bearish'):
            err_count_31 +=1
        if (len(response_llam.choices[0].message.content) != len("bearish")):
            err_count_32 += 1

In [ ]:
print(f'err_count1 Phi4:mini={err_count_1}, error non sensitive case={err_count_11}, err_length={err_count_12}')
print(f'err_count2 qwen2.5:7b={err_count_2}, error non sensitive case={err_count_21}, err_length={err_count_22}')
print(f'err_count3 llama3.1:8b={err_count_3}, error non sensitive case={err_count_31}, err_length={err_count_32}')

From the tests I made, it seemed the more you ask the same question to the model the quicker it tends to answer, I did many tests the test for 4 models:
- gemma3:270M, the quickest by far (the slowest were ~1s per attempt) but does not give the bests results as the classification on the same message can change with the same prompt. This is not a reliable behaviour
- qwen2.5:7b, the slowest (best time so far ~10s/attempt, worst 40s), but provides nuances, and gives clear explanation, why the message is considered to be bearish. the classification is consistent
- phi4-mini, is quicker than qwen on the same prompt( best time so far ~4s/attempt, worst 17s) it also gives clear consistent detailed explanation of the classification made
- llama3.1:8b, on the few tests I made, Llama3.1:8b tend to have the same level of accuracy as the two previous, and execution time comparable to the qwen2.5b model (worst )

Qwen2.5:7b has a larger context, but is slower than phi4-mini which produces comparable results. for the text we want to classify. Phi4-min also has the advantage of take less disk place and needs less computation power. 


I tried some prompt improvement to increase the execution.
1. I first change the role in the prompt: When the *role* changes from "user" to "financial expert", qwen model becomes impredictible giving feedbacks like :"好的，有什么我可以帮助你解答的问题吗？不论是关于知识提问、学习辅导、问题探讨还是其他方面的话题，我都会尽力提供帮助。请告诉我你的需求", there is no visible decrease in execution time for phi4 and the execution time skyrockets for llama3.1b (it ran for more than 5 minutes for one request without feedback)
2. I changed the prompt adding into it to just respond with one word "bullish", "bearish" or "neutral". In result the response of the 3 models became quicker, but the answers seemed random. as the message that was previously systematicaly bearish started to be classified as neutral or even bullish in some cases.

This new promtp produces random like results, so instead of asking it to just answer one word, I requested it to summarize the answer in one word "bullish", "bearish" or "neutral". after this little change, the feedback was quicker on the 3 models( less than 1s) and the classification was consistent. Llama was the only one to still sometimes gives phrases like 'I would classify this message as **bearish**.'. so I added I changed again the prompt as following "when the classification is done sumarize your answer in **just** one word 'bearish', 'bullish' or 'neutral', even thouhg the classification where correct, it sometimes (1/30 times) provided answers like **"bears"**. 

after the last change, I tested it 300 times on the same message for the 3 models. it provided the following results:

- err_count1=80, err_case1=4, err_length=1 -> Phi4:mini execution time (2min10,9s)
- err_count2=0, err_case2=0, err_length=0 -> qwen2.5:7b execution time (4min25.4s)
- err_count3=295, err_case3=74, err_length=60 -> llama3.1:8b execution time (10min33.9s)

From the previous result we are going to use qwen2.5:7b for the classification of the messages as it gives a good compromise in execution time and accuracy of the prediction.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def classify(tweet: str) -> str:
    resp = ollama.chat.completions.create(
        model="qwen2.5:7b",
        messages=[{"role": "user", "content": f"classify this message as a bullish, bearish or neutral, when the classification is done sumarize your answer in ONLY one word from: 'bearish', 'bullish' or 'neutral': {tweet}"}],
        temperature=0
    )
    return resp.choices[0].message.content.lower()

# Boucle concurrente
#with ThreadPoolExecutor(max_workers=4) as pool:
#    df_processed_btx_BR['sentiment_from_llm'] = list(pool.map(classify, df_processed_btx_BR['message_body'].tolist()))   
with ThreadPoolExecutor(max_workers=4) as pool:
    df_processed_btx_BR['sentiment_from_llm'] = list(tqdm(pool.map(classify, df_processed_btx_BR['message_body'].tolist())))


11672it [11:32:24,  2.43s/it]

The previous code took 8H without finishing the computation and provide feedback. I decided to run it on colab and download the results
meanwhile I am going to prepare the data corresponding to *bearmarket* and make it process by the same llm
we will focus on a periode after the pick.
this messages starts from the msg_ID=105355476 to msgID= 112010585
these messages are from file msg_001 up to file msg_004

I'll do the preprocessing and and teh word bitcoin



In [ ]:
bear_market_begin = 105355476
bear_market_end_slop = 112010585
df_bear_market = pd.concat([
    pd.read_csv('../data/msg/msg_001.csv'),
    pd.read_csv('../data/msg/msg_002.csv'),
    pd.read_csv('../data/msg/msg_003.csv'),
    pd.read_csv('../data/msg/msg_004.csv', engine='python'),# je n'ai pas pu loader les dernier fichier alors je me suis arreter à 4
    ], ignore_index=True)


C:\Users\xis\AppData\Local\Temp\ipykernel_21560\2958632168.py:4: DtypeWarning: Columns (0: message_id, 1: message_body) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('../data/msg/msg_001.csv'),
C:\Users\xis\AppData\Local\Temp\ipykernel_21560\2958632168.py:5: DtypeWarning: Columns (0: message_id) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('../data/msg/msg_002.csv'),
C:\Users\xis\AppData\Local\Temp\ipykernel_21560\2958632168.py:6: DtypeWarning: Columns (0: message_id) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('../data/msg/msg_003.csv'),# je n'ai pas pu loader les dernier fichier alors je me suis arreter à 4


In [23]:

df_bear_market = df_bear_market[df_bear_market['message_id'].str.fullmatch(r'\d+', na=False)].reset_index(drop=True)
df_bear_market['message_id'] = df_bear_market['message_id'].astype('int')
df_bear_market = df_bear_market[(df_bear_market['message_id']>=bear_market_begin) & (df_bear_market['message_id']<=bear_market_end_slop)].reset_index(drop=True)


In [24]:
df_bear_market['message_body']=df_bear_market['message_body'].str.replace("&#39;","'")
df_bear_market = df_bear_market[df_bear_market['message_body'].str.contains(r'\$btc|bitcoin', case=False)].reset_index(drop=True)
df_bear_market.head()
df_bear_market.shape 

(136574, 2)

In [25]:
df_bear_market.to_csv("../data/classification_Bearmarket.csv", index=False)

In [ ]:
#now based on the previous classification, we are going to fine tune the finbert classifier
#--- 0. Import of needed library ---
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
# --- 1. preparation of the data ---
#prepare les données pour le finetuning
# df doit avoir 'text' et 'label' : 0=bearish, 1=bullish, 2=neutral
df = pd.concat([df_processed_btx_BR, df_processed_btx_BM], ignore_index=True)   
texts = df['text'].tolist()
labels = df['label'].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [ ]:
# --- 2. Tokenizer ---
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(tokenize, batched=True)
val_ds = Dataset.from_dict({"text": val_texts, "label": val_labels}).map(tokenize, batched=True)


In [ ]:
# --- 3. Modèle ---
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Remapper les labels (FinBERT natif : 0=positive, 1=negative, 2=neutral)
#  en : 0=bearish, 1=bullish, 2=neutral
model.config.id2label = {0: "bearish", 1: "bullish", 2: "neutral"}
model.config.label2id = {"bearish": 0, "bullish": 1, "neutral": 2}

# --- 4. Entraînement ---
training_args = TrainingArguments(
    output_dir="./finbert_tweet_classifier",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./finbert_tweet_classifier")
tokenizer.save_pretrained("./finbert_tweet_classifier")   